# 🔍 AI Evaluation Error Analysis Toolkit

## Overview

This notebook provides a comprehensive data analysis toolkit for AI generated data. It helps you understand patterns in data, including sommon evaluation failures, identify common issues, and gain insights into how to improve your AI system's performance.



## Prerequisites

Before running this notebook, ensure you have:
- Evaluation data in the expected format (see Data Structure section below)
- OpenAI/Azure OpenAI credentials configured for LLM analysis
- Required Python packages installed

Let's get started! 🚀

# 🛠️ Setup and Configuration

## Environment Setup

This notebook requires several dependencies and configuration steps:

### 1. Install Required Packages
```bash
pip install matplotlib seaborn wordcloud plotly ipywidgets
```

### 2. Configure OpenAI/Azure OpenAI (for LLM analysis)
Create a `.env` file with your credentials:
```env
AZURE_OPENAI_API_KEY=your_api_key_here
AZURE_OPENAI_ENDPOINT=https://your-endpoint.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-02-01
AZURE_OPENAI_DEPLOYMENT_NAME=your_deployment_name
```



### 3. Enable Auto-reload
The cell below enables automatic reloading of modules, so changes to the error analyzer code are picked up automatically.

In [1]:
%load_ext autoreload
%autoreload 2

# 📦 Import Analysis Toolkit

Now we'll import our error analysis tools:
- **`ErrorAnalyzer`**: The main class that performs error pattern analysis
- **`load_evaluation_data_with_samples`**: Utility to load evaluation data with full conversation context

**Note**: The enhanced report now includes conversation data directly in `raw_imperfect_data`, so the interactive drill-down no longer requires separate conversation data.

In [2]:
from data_analyzer import DataAnalyzer

# 📊 Load and Explore Your Data

## Data Loading

We'll load data that contains both:
1. **Context**: Scores, pass/fail results, and detailed reasons
2. **Original conversation**: Full conversation data with queries and responses
3. **Metadata**

## Expected Data Structure

Your evaluation data should follow this structure:
```python
{
    "context": this is the analysis context, for error analysis this is the Evaluation output,
    "conversation": the conversation this context is attached to,
    "metadata": metadata
}
```

Let's load your data and examine its structure:

In [4]:
import json
with open("./data/data_analyzer_input_4o.json", "r") as f:
    data_analyzer_input = json.load(f)

print(f"loaded {len(data_analyzer_input)} entries")

loaded 585 entries


In [4]:
# filter data_analyzer_input to entries with metadata/score less than 3
data_analyzer_input_error = [
    entry for entry in data_analyzer_input if entry.get("metadata", {}).get("score", 0) < 3
]
print(f"filtered {len(data_analyzer_input_error)} entries with score < 3")

filtered 102 entries with score < 3


let's creating embeddings for this file

In [5]:
data_analyzer = DataAnalyzer()

In [ ]:
processed_file_base = "fails_only_loc_effect_4o_agents"
processed_file_path = f"./data/data_analyzer_input_{processed_file_base}_with_embeddings.json"
# processed_entries = data_analyzer.process_entries(input_files = ["./data/data_analyzer_input_41_mini_fails_only.json"], save_path=processed_file_path)

processed_entries = data_analyzer.process_entries(input_files = ["./data/data_analyzer_input_asst_rtPgXi5EDUqwIoGwdkgVQpb4.json","./data/data_analyzer_input_4o_asst_r5IWl11XLj74DHeKh9hbFriz.json"], save_path=processed_file_path)

print(f"processed {len(processed_entries)} entries")
print(processed_entries[0])

processed 185 entries
{'id': 0, 'context_summary': "This context concerns evaluating an agent's ability to resolve user intent in the travel domain. The user requested attractions near the Colosseum in Rome, but the agent failed by providing only a generic, non-specific response, highlighting deficiencies in meeting user expectations for relevant and detailed information.", 'conversation_summary': 'The conversation defines a WeatherAndMaps Assistant designed to provide weather updates and location-based services using Azure Maps and Weather APIs. Key entities include weather data functions (current, daily, hourly forecasts), location search tools (address, coordinates, points of interest), and utility functions (timezone, geolocation), all integrated to support user queries in the weather and mapping domain.', 'combined_summary': "Context: This context concerns evaluating an agent's ability to resolve user intent in the travel domain. The user requested attractions near the Colosseum i

## Run Analyzer
Cluster the evaluation results into insightful clusters.

In [7]:
with open(processed_file_path, "r") as f:
    processed_entries = json.load(f)
print(f"loaded {len(processed_entries)} processed entries")

loaded 185 processed entries


In [8]:
data_analyzer = DataAnalyzer()
clustering_method = "kmeans"  # or "hdbscan"
data_analysis_results = data_analyzer.analyze(processed_entries=processed_entries, clustering_method=clustering_method, num_clusters=4)

In [9]:
# save report as json
import json
report_path = f"./data/data_analysis_report_{processed_file_base}_{clustering_method}.json"
with open(report_path, "w") as f:
    json.dump(data_analysis_results, f, indent=4)

In [ ]:
data_analysis_results.keys()

In [ ]:
# data_analysis_results

# output format


### Data Analysis Report Structure

The object `data_analysis_results` is a dictionary with these top-level keys:

- `summary`: Overall stats
  - `total_entries`: Number of input entries analyzed
  - `unique_subcluster_labels`: Count of distinct subclusters
  - `total_clusters`: Number of top-level clusters
  - `clustering_method`: `llm` or `classic`
- `axes`: Names of the 2D coordinate axes (always `embeddings1`, `embeddings2`)
- `entries`: List of per-entry records
  - Each entry contains:
    - `id`: Entry identifier (original id or index)
    - `context_summary`: Context-focused summary text (or raw context fallback)
    - `conversation_summary`: Conversation summary (may be empty if no conversation)
    - `combined_summary`: Joined context + conversation summary
    - `subcluster_label`: Assigned subcluster label
    - `metadata`: Original metadata plus:
      - `conversation_turns`: Count of detected conversation turns
    - `coordinates`: 2D embedding coordinates `[x, y]`
- `subclusters`: Mapping subcluster_label -> object
  - `entry_ids`: List of entry ids in the subcluster
  - `count`: Number of entries
  - `coordinates`: Mean 2D position of its entries
  - `description`: Textual description (LLM / heuristic / classic)
- `clusters`: Mapping cluster_name -> object
  - `weight`: Total entries in all member subclusters
  - `subcluster_labels`: List of subcluster labels in the cluster
  - `subcluster_counts`: Per-subcluster entry counts
  - `description`: Textual description (LLM / heuristic / classic)
  - `coordinates`: Weighted average (by subcluster count) 2D position
- `llm_analysis`: Optional natural language bullet summary (None if disabled or no LLM)
- `classic_metadata`: Present only for `classic` method (e.g., vectorizer, subcluster_k)
- `raw`: The original input entries you supplied (unaltered)

#### Drill-Down Flow
1. Start at `clusters` to view thematic groups.
2. Expand a cluster to see its `subcluster_labels`.
3. Look up each label in `subclusters` to get member entry ids.
4. Fetch those entries from `entries` for detailed summaries & metadata.

#### Coordinates
- Subcluster and cluster coordinates are averages (cluster average is weighted by subcluster size).

Use these fields to build dashboards, interactive scatter plots, or hierarchical explorers.

# 📈 2D Cluster Visualizations

Below we visualize the embedding / coordinate space produced by the DataAnalyzer.

- Subcluster view: Each point represents a subcluster. Point size = number of entries.
- Entry view: Each point is an entry colored by its subcluster.

Axis labels come from `data_analysis_results['axes']`.

In [19]:
import json
with open("./data/data_analysis_report.json", "r") as f:
    data_analysis_results = json.load(f)

In [20]:
from utils import visualize_data_analyzer_2d

visualize_data_analyzer_2d(data_analysis_results)